# Gemini runs of the main analysis

**Run the cells from top to bottom.** If anything stops (internet, error, closing the notebook), run the same cell again: it continues where it stopped, with exactly the same batches.

All runs use **fixed generation settings**: temperature = 0, seed = 42, thinking budget = 8,192 tokens, **full review text** (no truncation), batches of 15 reviews formed **at random** (not in file order).

| Part | What | Reviews | Approx. cost |
|---|---|---|---|
| TEST | 1 tiny request to check key and model | 2 | ~0 |
| RUBRIC | the 300 human-reference reviews, prompt aligned with the annotators' rules (explicit-mention rule, no format examples) | 300 | < 1 USD |
| SINGLE | 2,000 random reviews, **one review per request** | 2,000 | ~5 USD |
| RUN1, RUN2, RUN3 | the whole corpus, original prompt, three independent runs with different random batches | 3 × 14,838 | ~20–25 USD each |
| POLARITY | the whole corpus, **sentiment-only prompt** (same batches as RUN1) | 14,838 | ~5 USD |

Needs `marakech.csv` (the collected file, 14,838 rows) in the same folder. Results go to `revision2_results.zip`. The API key is typed at run time and never saved.


In [ ]:
# ============ CELL 1 : INSTALL (only the first time) ============
%pip install -q google-genai pandas tqdm

In [ ]:
# ============ CELL 2 : SETTINGS ============
RUN_RUBRIC   = True
RUN_SINGLE   = True
RUN_MAIN     = [1, 2, 3]       # the three full-corpus runs
RUN_POLARITY = True

MODEL_NAME      = 'gemini-2.5-flash'
TEMPERATURE     = 0.0
GEN_SEED        = 42
THINKING_BUDGET = 8192
BATCH_SIZE      = 15
MAX_WORKERS     = 8      # parallel requests (does not change the results, only the speed)
MAX_TRIES       = 4
N_SINGLE        = 2000
FALLBACK_SINGLE = True   # re-send reviews of persistently failing batches one by one
RUN_SINGLE_300  = True   # the 300 human-reference reviews, one review per request (original prompt)
BUDGET_USD      = 100.0  # safety: every part stops automatically if the total spent reaches this amount

PRICE_INPUT  = 0.30   # USD per 1M input tokens
PRICE_OUTPUT = 2.50   # USD per 1M output tokens (thinking tokens are billed as output)

INPUT_FILE = 'marakech.csv'
OUT_DIR    = 'revision2_outputs'

In [ ]:
# ============ CELL 3 : IMPORTS + API KEY (typed, never saved) ============
import os, json, time, hashlib, platform, datetime, zipfile, getpass, random
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from google import genai
from google.genai import types
try:
    from tqdm import tqdm          # text progress bar: works in any Jupyter, no widget needed
except ImportError:
    tqdm = None

client = genai.Client(api_key=getpass.getpass('Paste your Gemini API key and press Enter: '),
                      http_options=types.HttpOptions(timeout=300_000))
CONFIG = types.GenerateContentConfig(temperature=TEMPERATURE, seed=GEN_SEED,
                                     thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET))
os.makedirs(OUT_DIR, exist_ok=True)
import google.genai as _g
print('OK - google-genai', _g.__version__, '| model:', MODEL_NAME)

In [ ]:
# ============ CELL 4 : LOAD DATA + CHECKS ============
df = pd.read_csv(INPUT_FILE)
assert len(df) == 14838, f'Expected 14,838 rows, found {len(df)} - wrong file?'
TEXT = df['text'].astype(str).str.strip().tolist()     # full text, whitespace trimmed only
ALL_IDS = list(range(len(df)))

HUMAN_IDS = [8023, 5833, 2542, 13482, 10116, 8678, 1669, 1713, 14604, 934, 2168, 4085, 10863, 6376, 6983, 11083, 6219, 1758, 10276, 1632, 12488, 2949, 2574, 3025, 3505, 9526, 4020, 216, 13493, 9124, 3816, 2060, 6071, 5479, 9159, 10622, 12468, 14536, 10132, 13580, 380, 12549, 1190, 11687, 10965, 2918, 1840, 1973, 9477, 248, 6116, 11155, 499, 3685, 13320, 1457, 12066, 13861, 4231, 5268, 150, 14307, 10418, 13747, 3345, 4039, 1158, 5737, 2158, 11549, 2712, 14203, 6661, 10869, 13539, 11720, 12664, 13160, 13581, 9500, 13203, 9334, 9443, 4542, 10385, 11521, 8319, 6528, 2299, 7820, 14127, 13803, 1380, 2597, 7469, 8031, 8214, 12079, 2351, 13465, 1349, 7390, 12200, 9206, 13474, 9620, 7041, 17, 5331, 7678, 14402, 7632, 6294, 4178, 10876, 11197, 7628, 13547, 2896, 4574, 13954, 13016, 3642, 242, 10279, 7727, 9631, 5364, 6337, 2538, 14767, 10922, 11140, 4100, 161, 9233, 4703, 5523, 9926, 6181, 4895, 9119, 13664, 2587, 5807, 14072, 3064, 1000, 5209, 2564, 5063, 3197, 121, 10038, 4354, 11472, 14835, 4409, 1145, 8522, 4831, 489, 8942, 8766, 11710, 5007, 1028, 108, 6708, 9740, 9463, 5845, 947, 2810, 9112, 6998, 12901, 10333, 11644, 942, 2577, 6580, 13740, 1452, 14115, 8358, 2243, 9925, 14095, 13616, 665, 10743, 7977, 1392, 7034, 6777, 1915, 3820, 6628, 1699, 3480, 10322, 2264, 8876, 11047, 2612, 2391, 9432, 9781, 8337, 12732, 2899, 3050, 10616, 676, 13805, 11318, 6851, 12225, 6873, 10914, 274, 4820, 13964, 12262, 8755, 7553, 14361, 327, 1790, 12712, 234, 1394, 13079, 1828, 12613, 12150, 13004, 10065, 9106, 9319, 267, 3953, 14160, 13732, 2546, 14466, 8424, 1221, 1831, 13700, 13263, 3977, 184, 3323, 2340, 12917, 5851, 378, 13637, 12761, 7509, 2658, 10627, 14234, 13492, 10396, 9184, 13322, 6328, 10339, 7717, 7289, 3581, 3483, 5354, 10851, 6287, 5359, 8929, 668, 10498, 11111, 2930, 5067, 1663, 7686, 6751, 10019, 6436, 7546, 4568, 4403, 10850, 5345, 6123, 3555, 2113, 5087, 8127]
HUMAN_FP  = ['5e168f4d', '97403cdc', '590d494b', 'b027b266', '77138c36', '9d185c48', '2209b793', '120602f4', '7b4402b2', '49ebfa75', 'ba328cc2', '09a3c64a', '385e650e', '2f2632c2', 'b0b9e4b3', '82527eaa', 'bfc8e031', '9dc0eb39', '6f315c5c', '59d20326', '3f8a339a', '34ccd2c4', 'c62feb8b', '3124c1db', 'cae1f1e0', '9081cce3', 'e8638795', '50eb6fc0', 'e9532022', 'ad414a0a', 'bad50589', '3a1591c8', '501c87bd', '9a16eb88', '022c6e5c', 'b8585b9b', '4d448f6a', 'dc4b2028', '3c9652d4', '87521986', 'b6f22f51', 'df0ce2d4', '73db37b7', '4dff2667', '27d6435b', '17338b8a', 'c7bed974', '36e4b361', '58938e8f', '931e727c', '1b0c5faf', '269bb8ed', '6872c9b9', 'ebc48e0d', 'd0a1a3b5', '8a6ea68f', '9c238ed9', 'ec5f272c', '73c12b82', '9a167655', '467f13aa', '84f6292e', '50c79ae4', '78a4fc7b', '1c134b5a', '03c52215', '903daea0', 'b8038ef5', 'daa3202d', '2ec1919d', '55605779', 'b855caf9', 'e0d870d2', '2612e2f5', '0846fe2f', 'aaad0673', '593c29a4', '17e947bd', 'f2f4c4b4', '605c30eb', 'ebbfac7b', '39d726a0', '1d4e504f', '9b2a6cac', 'd0d7e764', '43918228', '06fd651d', '12ee5a4c', '6f82f320', 'b00567eb', 'f6be4602', '73a3865f', '36a785e1', '8dcf6b1d', 'd86c6763', 'fe199ebb', 'c842022a', '73736561', 'f8e1dd77', '38a3a560', 'a3c39366', '25053e18', '40a5aab4', '03de5f9e', '02a408b2', 'e06e5bf2', 'a968a459', 'b191a02d', '8c11f004', '080f3411', '78763a8b', '4e1608a8', '2b700a51', '9c977e46', '6248570a', 'fafab820', 'f60d54d9', '591cb501', '923c1c07', '74dfdfdd', '61921ea9', 'f6f999c8', '8ea9c1b3', 'ab8e7d0a', '472ae392', '4decdd48', 'c65d705c', 'dc3211b3', 'd8ad4f7d', 'c95e4451', 'a6738a4b', 'fb9fd549', 'ce7bdceb', '4316c0b1', '2913e7ee', '47efa367', 'e517b3d3', 'b088a1dd', '43a9d1b2', '0eeef6fb', 'dd1b90e5', '4d2db8e7', 'cb1eadff', '692e0d7d', '553328de', '6ea9cc4d', 'c24fd46b', '3c0cda13', 'df0a4e56', '935b58f3', 'e148d8b7', 'a5420b9c', '9d979543', 'fd2409b3', 'ac2d9f24', '9d196f7c', '6390fc09', '568a2a06', 'bfb7febd', '7b67fd9b', 'f15f14b1', 'd3e6e325', '1dc56622', 'a438bd76', '04c42d04', 'bfebc4ad', '4f5c345b', 'baa41057', 'c0bb467a', 'ed613b89', '8d93d1ab', 'aa5db124', 'b284d2a9', '4599e3d9', 'e35ef321', '8a300a2f', '91b2906c', '2c9b4867', '31d83f73', '04c18104', 'ed83ef67', 'ff3f17b4', '9679d169', '11bfc657', 'a969fbd1', 'd956c993', 'cd5e16d7', 'ae37b583', 'a4800426', '8aad67d8', '614f57b0', 'c0edba77', 'a480148b', 'dd74939e', 'aede4017', '60d23dd9', 'd8126f56', 'c9c8f52d', 'e106c06e', 'de5d0efc', 'e7c8896a', 'dc4545e7', 'd2ce120a', '7410cf7b', '92cce2ac', '06dbd261', 'a7db105b', '9ae47524', 'c5bdb165', '1d5c31ce', '08e6e0f6', 'fecbed02', 'fe8a224e', '71b814c1', '618d2ef8', '29f18a43', 'fa8c7e42', '9a10a37d', '5c392de1', '8095271b', '502a0d9a', 'ca1c9084', '4aecc470', '6074b4fb', '27b4b7c2', '864dd560', 'aad6d1c0', '97fe22ab', 'e243d109', '031149d9', '76f014ec', 'ac3632c2', '44ca47e7', 'd181b15e', '8dc97a43', 'fed320e4', 'f9317644', 'e9cd56ea', '6a0b2481', '39cd529b', '92918f93', 'e8b71edf', '8eea913a', '88ac45b8', 'b15e38cf', 'f983246b', 'fb4f3e2f', '16d4b9ac', '93c234ce', '946f8ccd', 'a070d520', '4b612a36', '3faed75e', '75926a78', '1a2663fe', '8e3ed4b0', '8f31f757', 'cc561ba2', 'eb3eb116', '82b0669e', 'aca84925', '3938d6c3', 'c02e7d7d', 'd8413e14', '3780ea4e', '1be4c3f2', '01f5259f', '89f29535', 'b7f5e32e', '65a51980', '76e0d690', 'c98dae2f', 'bb8c049e', 'f171b074', 'cd15a974', '53168b56', '3ae3a849', '5c861d14', 'c37d9f55', 'f98d3519', 'bd1052c1', 'cbe8576c', '608233f1', 'c59682d4', '90ce77fb', '9b41ff10', '3b820659', 'e2188592', '6ff08e9a', '93035204', '1fa618fb', 'be75eaca', '55bdb342', 'db92b4a9', '06061b1c', 'ac0f91b4', 'ed683e7d', 'c6272bfb', '9c8f323d', '3b5f370a']
fp = [hashlib.md5(TEXT[i].encode('utf-8')).hexdigest()[:8] for i in HUMAN_IDS]
assert sum(a == b for a, b in zip(fp, HUMAN_FP)) == 300, 'The 300 human-reference reviews do not match - wrong file?'

SINGLE_IDS = sorted(random.Random(7).sample(ALL_IDS, N_SINGLE))
print('Checks OK: 14,838 reviews, 300 human-reference reviews found, 2,000 single-call reviews drawn.')

In [ ]:
# ============ CELL 5 : PROMPTS ============
# (a) ORIGINAL prompt - identical to the one used in the paper
def original_prompt(reviews_json):
    return """You are a tourism analyst. Analyze each TripAdvisor review and return ONLY a JSON array (same length and order as input). No explanation, no markdown.

For each review, return:
{
  "overall_sentiment": "Positive" | "Negative" | "Neutral",
  "sentiment_score": 1-5,
  "emotional_tone": "Joy" | "Anger" | "Fear" | "Disgust" | "Sadness" | "Surprise" | "Trust" | "Frustration" | "Neutral" | "Other",
  "aspects": {
    "safety": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "pricing": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "service": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "cleanliness": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "atmosphere": "Positive" | "Negative" | "Neutral" | "Not_mentioned"
  },
  "problems": ["Aggressive_vendors" | "Navigation_difficulty" | "Getting_lost" | "Bad_smell" | "Overcrowding" | "Poor_condition" | "High_prices" | "Poor_cleanliness" | "Scam" | "Safety" | "Other"],
  "economic_signals": {
    "price_perception": "Expensive" | "Fair" | "Cheap" | "Not_mentioned",
    "recommendation": "Yes" | "No" | "Not_mentioned",
    "revisit_intention": "Yes" | "No" | "Not_mentioned"
  },
  "keywords": ["keyword1", "keyword2", "keyword3"]
}

Rules:
- If no problems, return "problems": []
- Maximum 3 problems per review
- Maximum 3-5 keywords per review
- Translate any non-English problems/keywords into English
- Be consistent: same issue = same wording across all reviews
- Return ONLY the JSON array, nothing else

Examples:
"Beautiful place but shopkeepers are aggressive and everything is overpriced"
→ {"overall_sentiment": "Negative", "sentiment_score": 2, "emotional_tone": "Frustration", "aspects": {"safety": "Not_mentioned", "pricing": "Negative", "service": "Negative", "cleanliness": "Not_mentioned", "atmosphere": "Positive"}, "problems": ["Aggressive_vendors", "High_prices"], "economic_signals": {"price_perception": "Expensive", "recommendation": "Not_mentioned", "revisit_intention": "Not_mentioned"}, "keywords": ["beautiful", "aggressive", "overpriced"]}

"Amazing experience, felt very safe, would definitely come back!"
→ {"overall_sentiment": "Positive", "sentiment_score": 5, "emotional_tone": "Joy", "aspects": {"safety": "Positive", "pricing": "Not_mentioned", "service": "Not_mentioned", "cleanliness": "Not_mentioned", "atmosphere": "Positive"}, "problems": [], "economic_signals": {"price_perception": "Not_mentioned", "recommendation": "Yes", "revisit_intention": "Yes"}, "keywords": ["amazing", "safe", "come back"]}

Reviews:
""" + reviews_json + """

Return ONLY the JSON array."""

# (b) SENTIMENT-ONLY prompt (same wording and examples, one dimension)
EX1_TEXT = "Beautiful place but shopkeepers are aggressive and everything is overpriced"
EX2_TEXT = "Amazing experience, felt very safe, would definitely come back!"
EX1 = {"overall_sentiment": "Negative", "sentiment_score": 2, "emotional_tone": "Frustration",
       "aspects": {"safety": "Not_mentioned", "pricing": "Negative", "service": "Negative", "cleanliness": "Not_mentioned", "atmosphere": "Positive"},
       "problems": ["Aggressive_vendors", "High_prices"],
       "economic_signals": {"price_perception": "Expensive", "recommendation": "Not_mentioned", "revisit_intention": "Not_mentioned"}}
EX2 = {"overall_sentiment": "Positive", "sentiment_score": 5, "emotional_tone": "Joy",
       "aspects": {"safety": "Positive", "pricing": "Not_mentioned", "service": "Not_mentioned", "cleanliness": "Not_mentioned", "atmosphere": "Positive"},
       "problems": [],
       "economic_signals": {"price_perception": "Not_mentioned", "recommendation": "Yes", "revisit_intention": "Yes"}}

SCHEMA = {
 'sentiment': '  "overall_sentiment": "Positive" | "Negative" | "Neutral"',
 'score':     '  "sentiment_score": 1-5',
 'emotion':   '  "emotional_tone": "Joy" | "Anger" | "Fear" | "Disgust" | "Sadness" | "Surprise" | "Trust" | "Frustration" | "Neutral" | "Other"',
 'aspects':   """  "aspects": {
    "safety": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "pricing": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "service": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "cleanliness": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "atmosphere": "Positive" | "Negative" | "Neutral" | "Not_mentioned"
  }""",
 'problems':  '  "problems": ["Aggressive_vendors" | "Navigation_difficulty" | "Getting_lost" | "Bad_smell" | "Overcrowding" | "Poor_condition" | "High_prices" | "Poor_cleanliness" | "Scam" | "Safety" | "Other"]',
 'economic':  """  "economic_signals": {
    "price_perception": "Expensive" | "Fair" | "Cheap" | "Not_mentioned",
    "recommendation": "Yes" | "No" | "Not_mentioned",
    "revisit_intention": "Yes" | "No" | "Not_mentioned"
  }""",
}
KEY = {'sentiment': 'overall_sentiment', 'score': 'sentiment_score', 'emotion': 'emotional_tone',
       'aspects': 'aspects', 'problems': 'problems', 'economic': 'economic_signals'}
EXTRA_RULES = {
 'problems': ['If no problems, return "problems": []', 'Maximum 3 problems per review',
              'Translate any non-English problems into English',
              'Be consistent: same issue = same wording across all reviews'],
}

def single_prompt(dim, reviews_json):
    k = KEY[dim]
    rules = EXTRA_RULES.get(dim, []) + ['Return ONLY the JSON array, nothing else']
    ex1 = json.dumps({k: EX1[k]}, ensure_ascii=False)
    ex2 = json.dumps({k: EX2[k]}, ensure_ascii=False)
    return ("You are a tourism analyst. Analyze each TripAdvisor review and return ONLY a JSON array "
            "(same length and order as input). No explanation, no markdown.\n\n"
            "For each review, return:\n{\n" + SCHEMA[dim] + "\n}\n\n"
            "Rules:\n" + "\n".join("- " + r for r in rules) + "\n\n"
            "Examples:\n\"" + EX1_TEXT + "\"\n→ " + ex1 + "\n\n\"" + EX2_TEXT + "\"\n→ " + ex2 + "\n\n"
            "Reviews:\n" + reviews_json + "\n\nReturn ONLY the JSON array.")

def polarity_prompt(reviews_json):
    return single_prompt('sentiment', reviews_json)

# (c) RUBRIC-ALIGNED prompt: what the annotators received (label lists + explicit-mention rule), no format examples
def rubric_prompt(reviews_json):
    p = original_prompt(reviews_json)
    head, rest = p.split('Examples:', 1)
    tail = rest[rest.index('Reviews:'):]
    rule = ('- Recommendation and revisit intention: answer "Yes" or "No" ONLY when the reviewer states it explicitly '
            '(e.g. "I recommend", "we will come back"); if it is only implied by the tone, answer "Not_mentioned"\n')
    head = head.replace('- Return ONLY the JSON array, nothing else', rule + '- Return ONLY the JSON array, nothing else')
    return head + tail

print(rubric_prompt('["example review"]')[-900:])

In [ ]:
# ============ CELL 6 : ENGINE ============
LOG_FILE = os.path.join(OUT_DIR, 'token_log.csv')

def call_batch(part, batch_id, ids, prompt_fn):
    texts = [TEXT[i] for i in ids]
    reviews_json = json.dumps(texts, ensure_ascii=False)
    prompt = prompt_fn(reviews_json)
    logs = []
    for attempt in range(1, MAX_TRIES + 1):
        t0 = time.time()
        rec = {'part': part, 'batch_id': batch_id, 'attempt': attempt, 'n_reviews': len(ids),
               'input_chars': len(reviews_json), 'prompt_tokens': 0, 'output_tokens': 0, 'thinking_tokens': 0,
               'seconds': 0, 'ok': False, 'error': '', 'model_version': '',
               'timestamp': datetime.datetime.now().isoformat(timespec='seconds')}
        try:
            resp = client.models.generate_content(model=MODEL_NAME, contents=prompt, config=CONFIG)
            u = resp.usage_metadata
            rec.update(prompt_tokens=u.prompt_token_count or 0, output_tokens=u.candidates_token_count or 0,
                       thinking_tokens=u.thoughts_token_count or 0, model_version=str(resp.model_version or ''))
            content = (resp.text or '').strip().replace('```json', '').replace('```', '').strip()
            data = json.loads(content)
            if len(ids) == 1 and isinstance(data, dict):
                data = [data]
            if not isinstance(data, list) or len(data) != len(ids):
                raise ValueError(f'Length mismatch: expected {len(ids)}')
            rec.update(ok=True, seconds=round(time.time() - t0, 2)); logs.append(rec)
            return data, logs
        except Exception as e:
            rec.update(error=str(e)[:200], seconds=round(time.time() - t0, 2)); logs.append(rec)
            time.sleep(5 * attempt)
    return None, logs

def spent():
    if not os.path.exists(LOG_FILE): return 0.0
    g = pd.read_csv(LOG_FILE)
    return g.prompt_tokens.sum() / 1e6 * PRICE_INPUT + (g.output_tokens.sum() + g.thinking_tokens.sum()) / 1e6 * PRICE_OUTPUT

def make_batches(ids, batch_size, order_seed):
    ids = list(ids)
    random.Random(order_seed).shuffle(ids)          # random batch composition, reproducible
    return [ids[b:b + batch_size] for b in range(0, len(ids), batch_size)]

def run_part(part, ids, prompt_fn, order_seed, batch_size=BATCH_SIZE):
    out_file = os.path.join(OUT_DIR, f'{part}.csv')
    batches = make_batches(ids, batch_size, order_seed)
    done = set(pd.read_csv(out_file)['row_id']) if os.path.exists(out_file) else set()
    todo = [(k, b) for k, b in enumerate(batches) if not set(b) <= done]
    print(f'\n=== {part}: {len(ids)} reviews in {len(batches)} batches | batches left: {len(todo)} ===')
    bar = tqdm(total=len(todo), desc=part) if tqdm else None
    failed = 0
    step = MAX_WORKERS * 5
    for s in range(0, len(todo), step):
        if spent() >= BUDGET_USD:
            print(f'STOPPED: budget of {BUDGET_USD} USD reached (spent {spent():.2f}). Nothing is lost; raise BUDGET_USD to continue.')
            break
        chunk = todo[s:s + step]
        rows, logs = [], []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = {ex.submit(call_batch, part, k, b, prompt_fn): (k, b) for k, b in chunk}
            for f in as_completed(futs):
                k, b = futs[f]
                data, lg = f.result(); logs.extend(lg)
                if data is None:
                    failed += 1
                else:
                    for pos, (i, res) in enumerate(zip(b, data)):
                        rows.append({'row_id': i, 'batch_id': k, 'pos_in_batch': pos, 'raw_json': json.dumps(res, ensure_ascii=False)})
                if bar: bar.update(1)
        if rows:
            pd.DataFrame(rows).to_csv(out_file, mode='a', header=not os.path.exists(out_file), index=False)
        pd.DataFrame(logs).to_csv(LOG_FILE, mode='a', header=not os.path.exists(LOG_FILE), index=False)
    if bar: bar.close()
    # FALLBACK: a batch that failed MAX_TRIES times (usually the model returned the wrong number of answers)
    # is re-sent one review per request, so that no review is left without an answer.
    if FALLBACK_SINGLE and batch_size > 1 and spent() < BUDGET_USD:
        done = set(pd.read_csv(out_file)['row_id']) if os.path.exists(out_file) else set()
        left = [(k, b) for k, b in enumerate(batches) if not set(b) <= done]
        if left:
            print(f'{part}: {len(left)} batches still missing -> re-sent one review per request')
            rows, logs = [], []
            items = [(k, i) for k, b in left for i in b if i not in done]
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
                futs = {ex.submit(call_batch, part + '_fallback', f'{k}_single', [i], prompt_fn): (k, i) for k, i in items}
                for f in as_completed(futs):
                    k, i = futs[f]; data, lg = f.result(); logs.extend(lg)
                    if data is not None:
                        rows.append({'row_id': i, 'batch_id': f'{k}_single', 'pos_in_batch': 0, 'raw_json': json.dumps(data[0], ensure_ascii=False)})
            if rows: pd.DataFrame(rows).to_csv(out_file, mode='a', header=False, index=False)
            pd.DataFrame(logs).to_csv(LOG_FILE, mode='a', header=not os.path.exists(LOG_FILE), index=False)
    n = len(pd.read_csv(out_file)) if os.path.exists(out_file) else 0
    print(f'{part}: {n}/{len(ids)} done' + (' | some reviews still missing -> run this cell again' if n < len(ids) else '') + f' | total spent so far: {spent():.2f} USD')

In [ ]:
# ============ CELL 7 : TEST (2 reviews, almost free) ============
data, lg = call_batch('TEST', 'test', HUMAN_IDS[:2], original_prompt)
print('Result OK' if data else 'FAILED: ' + lg[-1]['error'])
print({k: lg[-1][k] for k in ['prompt_tokens', 'output_tokens', 'thinking_tokens', 'seconds', 'model_version']})
if data: print(json.dumps(data[0], ensure_ascii=False)[:300])

In [ ]:
# ============ CELL 8 : RUBRIC (300) + SINGLE (2,000 one per request) ~ 30-45 min ============
if RUN_RUBRIC:
    run_part('rubric_300', HUMAN_IDS, rubric_prompt, order_seed=300)
if RUN_SINGLE:
    run_part('single_call', SINGLE_IDS, original_prompt, order_seed=0, batch_size=1)

In [ ]:
# ============ CELL 8b : CHECK BEFORE THE BIG RUNS (free) ============
# Verifies that the answers are complete and readable, and estimates the cost of one full-corpus run.
r = pd.read_csv(os.path.join(OUT_DIR, 'rubric_300.csv'))
J = [json.loads(x) for x in r.raw_json]
ok_sent = sum(str(j.get('overall_sentiment')) in ('Positive', 'Negative', 'Neutral') for j in J)
print('rubric_300 answers:', r.row_id.nunique(), '/ 300 | valid sentiment labels:', ok_sent)
print(pd.Series([j.get('overall_sentiment') for j in J]).value_counts().to_dict())
g = pd.read_csv(LOG_FILE); g = g[g.part == 'rubric_300']
c300 = g.prompt_tokens.sum() / 1e6 * PRICE_INPUT + (g.output_tokens.sum() + g.thinking_tokens.sum()) / 1e6 * PRICE_OUTPUT
print(f'cost of these 300 reviews: {c300:.2f} USD  ->  ONE full run of 14,838 reviews ~ {c300 * 14838 / 300:.0f} USD')
print('(the 300 reviews are longer than average, so the real cost should be lower). Total spent so far:', round(spent(), 2), 'USD')
print('If the numbers look fine, continue with CELL 9.')

In [ ]:
# ============ CELL 9 : THREE FULL-CORPUS RUNS (~1-2 hours each) ============
for r in RUN_MAIN:
    run_part(f'run{r}', ALL_IDS, original_prompt, order_seed=100 + r)

In [ ]:
# ============ CELL 10 : SENTIMENT-ONLY PROMPT (same batches as RUN1) ============
if RUN_POLARITY:
    run_part('polarity_only', ALL_IDS, polarity_prompt, order_seed=101)

In [ ]:
# ============ CELL 12 : THE 300 HUMAN-REFERENCE REVIEWS, ONE REVIEW PER REQUEST (~1.5 USD, ~10 min) ============
if RUN_SINGLE_300:
    run_part('single_300', HUMAN_IDS, original_prompt, order_seed=0, batch_size=1)

In [ ]:
# ============ CELL 11 : SUMMARY + ZIP ============
expected = {'rubric_300': 300, 'single_call': N_SINGLE, 'single_300': 300, 'run1': 14838, 'run2': 14838, 'run3': 14838, 'polarity_only': 14838}
log = pd.read_csv(LOG_FILE); log = log[log.part != 'TEST']
rows = []
for part, n_exp in expected.items():
    f = os.path.join(OUT_DIR, f'{part}.csv')
    n_done = pd.read_csv(f).row_id.nunique() if os.path.exists(f) else 0
    g = log[log.part.isin([part, part + '_fallback'])]
    cost = g.prompt_tokens.sum() / 1e6 * PRICE_INPUT + (g.output_tokens.sum() + g.thinking_tokens.sum()) / 1e6 * PRICE_OUTPUT
    rows.append({'part': part, 'done': f'{n_done}/{n_exp}', 'requests': len(g), 'failed_attempts': int((~g.ok.astype(bool)).sum()),
                 'input_tok': int(g.prompt_tokens.sum()), 'output_tok': int(g.output_tokens.sum()), 'thinking_tok': int(g.thinking_tokens.sum()),
                 'minutes_api': round(g.seconds.sum() / 60, 1), 'cost_usd': round(cost, 3)})
summary = pd.DataFrame(rows); print(summary.to_string(index=False))
print('\nTOTAL estimated cost (USD):', round(summary.cost_usd.sum(), 2), '-> compare with the AI Studio billing page')
info = {'run_date': datetime.datetime.now().isoformat(timespec='seconds'), 'model': MODEL_NAME,
        'model_versions_seen': sorted(set(log.model_version.dropna().astype(str))),
        'library': 'google-genai ' + _g.__version__, 'python': platform.python_version(),
        'temperature': TEMPERATURE, 'seed': GEN_SEED, 'thinking_budget': THINKING_BUDGET,
        'batch_size': BATCH_SIZE, 'workers': MAX_WORKERS, 'max_tries': MAX_TRIES, 'input': 'full text, whitespace trimmed',
        'batch_order_seeds': {'run1': 101, 'run2': 102, 'run3': 103, 'polarity_only': 101, 'rubric_300': 300},
        'prices_used': {'input': PRICE_INPUT, 'output': PRICE_OUTPUT}}
json.dump(info, open(os.path.join(OUT_DIR, 'run_info.json'), 'w'), indent=2)
summary.to_csv(os.path.join(OUT_DIR, 'summary.csv'), index=False)
with zipfile.ZipFile('revision2_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(OUT_DIR):
        z.write(os.path.join(OUT_DIR, fn), fn)
print('\nDONE -> revision2_results.zip (no API key inside). Send this file.')